# 1차 Formulation

In [4]:
pip install gurobipy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
from gurobipy import Model, GRB

foods_list = [
    {
        "name": "삶은계란",
        "weight": 50,
        "cal": 71,
        "protein": 6,
        "fat": 5,
        "carb": 0.5,
        "price": 500,
    },
    {
        "name": "잡채",
        "weight": 75,
        "cal": 257,
        "protein": 2,
        "fat": 5,
        "carb": 51,
        "price": 1495,
    },
    {
        "name": "사과",
        "weight": 150,
        "cal": 63.8,
        "protein": 0.5,
        "fat": 0.2,
        "carb": 15,
        "price": 3400,
    },
    {
        "name": "제육볶음",
        "weight": 200,
        "cal": 430,
        "protein": 25,
        "fat": 30,
        "carb": 15,
        "price": 3500,
    },
    {
        "name": "두부",
        "weight": 100,
        "cal": 92.9,
        "protein": 9.4,
        "fat": 5.3,
        "carb": 1.9,
        "price": 500,
    },
    {
        "name": "물만두",
        "weight": 370,
        "cal": 696,
        "protein": 29,
        "fat": 32,
        "carb": 73,
        "price": 4500,
    },
    {
        "name": "고등어 구이",
        "weight": 100,
        "cal": 272,
        "protein": 24,
        "fat": 19,
        "carb": 1,
        "price": 2500,
    },
    {
        "name": "돼지고기 김치찌개",
        "weight": 460,
        "cal": 240,
        "protein": 24,
        "fat": 8,
        "carb": 18,
        "price": 7410,
    },
    {
        "name": "고추참치",
        "weight": 100,
        "cal": 142,
        "protein": 10,
        "fat": 6,
        "carb": 12,
        "price": 3300,
    },
    {
        "name": "차돌된장찌개",
        "weight": 460,
        "cal": 280,
        "protein": 17,
        "fat": 12,
        "carb": 26,
        "price": 7000,
    },
    {
        "name": "카레",
        "weight": 200,
        "cal": 171,
        "protein": 5,
        "fat": 7,
        "carb": 22,
        "price": 1980,
    },
    {
        "name": "쌀밥",
        "weight": 210,
        "cal": 297,
        "protein": 5,
        "fat": 1,
        "carb": 67,
        "price": 1000,
    },
    {
        "name": "참치마요 삼각김밥",
        "weight": 110,
        "cal": 180,
        "protein": 5,
        "fat": 8,
        "carb": 22,
        "price": 1700,
    },
    {
        "name": "삶은감자",
        "weight": 200,
        "cal": 176.76,
        "protein": 3.72,
        "fat": 0.2,
        "carb": 40.02,
        "price": 750,
    },
    {
        "name": "낫또",
        "weight": 40,
        "cal": 89,
        "protein": 7,
        "fat": 5,
        "carb": 4,
        "price": 700,
    },
    {
        "name": "계란말이",
        "weight": 100,
        "cal": 133,
        "protein": 10,
        "fat": 9,
        "carb": 3,
        "price": 1000,
    },
    {
        "name": "떡갈비",
        "weight": 100,
        "cal": 318,
        "protein": 12,
        "fat": 26,
        "carb": 9,
        "price": 2200,
    },
]

# 상수 정의
MIN_CAL = 2000  # 하루 최소 섭취 칼로리
MAX_CAL = 2500  # 하루 최대 섭취 칼로리
CARB_CAL = 4  # 탄수화물 1g당 칼로리
PROTEIN_CAL = 4  # 단백질 1g당 칼로리
FAT_CAL = 9  # 지방 1g당 칼로리

# 데이터 전처리
food_names = [f["name"] for f in foods_list]
foods_dict = {f["name"]: f for f in foods_list}

# 모델 생성
m = Model("diet_1st_formulation")

# 결정 변수 정의
b = m.addVars(food_names, name="breakfast", vtype=GRB.CONTINUOUS)
l = m.addVars(food_names, name="lunch", vtype=GRB.CONTINUOUS)
d = m.addVars(food_names, name="dinner", vtype=GRB.CONTINUOUS)

# 목적 함수 정의 - 총 단백질 섭취량 최대화
m.setObjective(
    sum(foods_dict[f]["protein"] * (b[f] + l[f] + d[f]) for f in food_names),
    GRB.MAXIMIZE,
)

totals = {
    "cal": sum(foods_dict[f]["cal"] * (b[f] + l[f] + d[f]) for f in food_names),
    "carb": sum(foods_dict[f]["carb"] * (b[f] + l[f] + d[f]) for f in food_names),
    "protein": sum(foods_dict[f]["protein"] * (b[f] + l[f] + d[f]) for f in food_names),
    "fat": sum(foods_dict[f]["fat"] * (b[f] + l[f] + d[f]) for f in food_names),
    "price": sum(foods_dict[f]["price"] * (b[f] + l[f] + d[f]) for f in food_names),
}

# 공통 제약식 (Common Constraints)
# c-1. 하루 최소 칼로리: 2000 kcal 이상
m.addConstr(totals["cal"] >= MIN_CAL, "c_min_calories")

# c-2. 탄수화물 비율: 총 칼로리의 55% 이상
m.addConstr(totals["carb"] * CARB_CAL >= totals["cal"] * 0.55, "c_carb_min_ratio")

# c-3. 탄수화물 비율: 총 칼로리의 65% 이하
m.addConstr(totals["carb"] * CARB_CAL <= totals["cal"] * 0.65, "c_carb_max_ratio")

# c-4. 단백질 비율: 총 칼로리의 7% 이상
m.addConstr(
    totals["protein"] * PROTEIN_CAL >= totals["cal"] * 0.07, "c_protein_min_ratio"
)

# c-5.단백질 비율: 총 칼로리의 20% 이하
m.addConstr(
    totals["protein"] * PROTEIN_CAL <= totals["cal"] * 0.20, "c_protein_max_ratio"
)

# c-6. 지방 비율: 총 칼로리의 15% 이상
m.addConstr(totals["fat"] * FAT_CAL >= totals["cal"] * 0.15, "c_fat_min_ratio")

# c-7. 지방 비율: 총 칼로리의 30% 이하
m.addConstr(totals["fat"] * FAT_CAL <= totals["cal"] * 0.30, "c_fat_max_ratio")

# 개인별 제약식 (Personal Constraints)
# p-1. 하루 최대 칼로리: 2500 kcal 이하
m.addConstr(totals["cal"] <= MAX_CAL, "p_max_cal")

# p-2. 각 끼니에 최소 1인분 이상 섭취
m.addConstr(sum(b[f] for f in food_names) >= 1, "p_breakfast_at_least_one")
m.addConstr(sum(l[f] for f in food_names) >= 1, "p_lunch_at_least_one")
m.addConstr(sum(d[f] for f in food_names) >= 1, "p_dinner_at_least_one")

# 최적화 실행
m.optimize()

# 결과 출력
if m.status == GRB.OPTIMAL:
    # 헤더
    print("\n" + "=" * 70)
    print("  My 1 Day Diet Program: 1st Formulation")
    print("=" * 70)
    print(f"\n  목적함수 값 (총 단백질 섭취량): {m.objVal:.2f}g\n")
    print("-" * 70 + "\n")

    # 아침 식사
    print("=" * 70)
    print(f"  아침 식사")
    print("=" * 70)
    b_cal_total = 0
    b_protein_total = 0
    b_price_total = 0
    for f in food_names:
        if b[f].X > 0.0001:
            print(f"  • {f}: {b[f].X:.2f}인분")
            b_cal_total += foods_dict[f]["cal"] * b[f].X
            b_protein_total += foods_dict[f]["protein"] * b[f].X
            b_price_total += foods_dict[f]["price"] * b[f].X
    print("-" * 70)
    print(
        f"  칼로리: {b_cal_total:.2f}kcal | 단백질: {b_protein_total:.2f}g | 비용: {b_price_total:,.0f}원\n"
    )

    # 점심 식사
    print("=" * 70)
    print(f"  점심 식사")
    print("=" * 70)
    l_cal_total = 0
    l_protein_total = 0
    l_price_total = 0
    for f in food_names:
        if l[f].X > 0.0001:
            print(f"  • {f}: {l[f].X:.2f}인분")
            l_cal_total += foods_dict[f]["cal"] * l[f].X
            l_protein_total += foods_dict[f]["protein"] * l[f].X
            l_price_total += foods_dict[f]["price"] * l[f].X
    print("-" * 70)
    print(
        f"  칼로리: {l_cal_total:.2f}kcal | 단백질: {l_protein_total:.2f}g | 비용: {l_price_total:,.0f}원\n"
    )

    # 저녁 식사
    print("=" * 70)
    print(f"  저녁 식사")
    print("=" * 70)
    d_cal_total = 0
    d_protein_total = 0
    d_price_total = 0
    for f in food_names:
        if d[f].X > 0.0001:
            print(f"  • {f}: {d[f].X:.2f}인분")
            d_cal_total += foods_dict[f]["cal"] * d[f].X
            d_protein_total += foods_dict[f]["protein"] * d[f].X
            d_price_total += foods_dict[f]["price"] * d[f].X
    print("-" * 70)
    print(
        f"  칼로리: {d_cal_total:.2f}kcal | 단백질: {d_protein_total:.2f}g | 비용: {d_price_total:,.0f}원\n"
    )

    # 전체 합계
    total_cal = b_cal_total + l_cal_total + d_cal_total
    total_protein = b_protein_total + l_protein_total + d_protein_total
    total_price = b_price_total + l_price_total + d_price_total

    print("=" * 70)
    print("  식단 최적화 결과")
    print("=" * 70 + "\n")
    print(f"  • 총 칼로리: {total_cal:.2f}kcal")
    print(f"  • 총 단백질: {total_protein:.2f}g")
    print(f"  • 총 예산: {total_price:,.0f}원")
    print("\n" + "-" * 70)

    # 영양소 비율 검증
    carb_cal = totals["carb"].getValue() * CARB_CAL
    protein_cal = totals["protein"].getValue() * PROTEIN_CAL
    fat_cal = totals["fat"].getValue() * FAT_CAL

else:
    print(f"\n최적해를 찾을 수 없습니다. (Status: {m.status})")

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 11 rows, 51 columns and 456 nonzeros
Model fingerprint: 0x5d3347e6
Coefficient statistics:
  Matrix range     [8e-01, 7e+02]
  Objective range  [5e-01, 3e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 2e+03]
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 10 rows, 52 columns, 406 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.5296017e+02   3.902192e+02   0.000000e+00      0s
      11    1.2500000e+02   0.000000e+00   0.000000e+00      0s

Solved in 11 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.250000000e+02

  My 1 Day Diet Program: 1st Formulation

  목적함수 값 (총 단백질 섭취량): 125.00g

----------------------------------------------------------------------

  아침 식사
  • 사과: 1.00인분
-----